# Collaborator Figure Workbook — v1.1 Figures

## 1. Load the production dashboard contexts

The public access point for collaborators should be the same context loaders used by the dashboard.

- Crime: `load_crime_dashboard_context()`
- CAD/calls: `load_dashboard_context()`
- UOF/OIS: `load_uof_dashboard_context()`
- Population: already exposed through the crime and calls contexts, or directly through `load_dashboard_population()`

Avoid reading raw parquet files directly unless you are debugging the snapshot layer itself. The context loaders apply the production cleaning, classification, geography, and derived-data rules.

The cell below is a mandatory setup cell for notebooks not in the repo root (in a folder), copy paste it into new notebooks you make.

In [1]:
from pathlib import Path
import sys

# -------------------------------------------------------------------
# Locate repository root robustly whether Jupyter starts from repo root
# or from the notebooks directory.
# -------------------------------------------------------------------

cwd = Path.cwd().resolve()

repo_candidates = [cwd, *cwd.parents]

REPO_ROOT = next(
    (
        path
        for path in repo_candidates
        if (path / "dashboard").is_dir()
        and (path / "app.py").exists()
    ),
    None,
)

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate repository root. "
        "Expected to find app.py and dashboard/."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

Repository root: C:\Users\benca\code\PersonalPythonProjects\SPDCallDashboard


In [2]:
from dashboard.crime_dashboard_data import load_crime_dashboard_context
from dashboard.spd_dashboard_data import load_dashboard_context
from dashboard.uof_dashboard_data import load_uof_dashboard_context
from dashboard.population_dashboard_data import load_dashboard_population

crime_context = load_crime_dashboard_context()
calls_context = load_dashboard_context()
uof_context = load_uof_dashboard_context()

neighborhood_population, city_population, population_metadata = load_dashboard_population()

### Context keys at a glance

**Crime context**

- `df` — full classified QA snapshot, including explicitly excluded rows.
- `valid_time` — authoritative analytical crime population with valid offense IDs/dates; includes offenses without map coordinates.
- `mappable_events` — analytical offenses with usable Seattle coordinates.
- `event_mcpp` — mappable offenses joined to spatial MCPP geography.
- `unmappable_events` — analytical offenses that cannot be represented as map points but may still contribute to analytical totals.
- `mcpp_boundaries` — MCPP polygons.
- `neighborhood_population` — calibrated MCPP population table.
- `city_population` — direct Seattle population estimate.
- `population_metadata` — ACS vintage/method metadata.

**Calls context**

- `df` — prepared dispatch-row snapshot.
- `valid_time` — dispatch rows with usable event IDs and queued timestamps.
- `event_mcpp` — coordinate-valid calls spatially matched to MCPPs.
- `response_analysis` — authoritative **event-level qualified response-time** table.
- `mcpp_boundaries` — MCPP polygons.
- `neighborhood_population`, `city_population`, `population_metadata` — population denominators and provenance.

**UOF context**

- `df` — full prepared UOF snapshot.
- `ois_events` — derived OIS events at the production day + normalized beat grain.
- `ois_rows` — source rows whose incident type matches OIS.
- `latest_available_date` — latest date in the UOF stream.
- `ois_rows_missing_event_date` — QA count for OIS rows that cannot form an event date.

In [3]:
print("Crime context keys:", sorted(crime_context))
print("Calls context keys:", sorted(calls_context))
print("UOF context keys:", sorted(uof_context))

Crime context keys: ['city_population', 'df', 'event_mcpp', 'event_mcpp_lookup', 'mappable_events', 'mcpp_boundaries', 'metadata', 'neighborhood_population', 'population_metadata', 'unmappable_events', 'valid_time', 'years_observed']
Calls context keys: ['city_population', 'df', 'event_mcpp', 'event_mcpp_lookup', 'mappable_events', 'mcpp_boundaries', 'metadata', 'neighborhood_population', 'population_metadata', 'response_analysis', 'valid_time', 'years_observed']
UOF context keys: ['df', 'latest_available_date', 'metadata', 'ois_events', 'ois_rows', 'ois_rows_missing_event_date']


## 2. Shared analysis-period helpers

For production-aligned analysis, use the shared period helpers rather than manually inventing date windows.

- `get_analysis_bounds(latest)` gives the selectable latest calendar-year domain.
- `get_previous_period(start, end, history_bounds=...)` gives the immediately adjacent equal-length comparison period.
- Crime filtering should use `filter_crime_records()` so category/subcategory/neighborhood/date behavior matches the dashboard.

In [4]:
import pandas as pd

from dashboard.analysis_windows import (
    get_analysis_bounds,
    get_history_bounds,
    get_previous_period,
)
from dashboard.crime_filters import filter_crime_records

## 3. Overall crime count KPI (Assignee: Ben)

**Authoritative source:** `crime_context["valid_time"]`

**Grain:** distinct `offense_id`

**Date field:** `offense_date`

Why `valid_time`? Citywide crime totals must include analytical offenses even when coordinates are missing or invalid. Do **not** count map points or use `event_mcpp` as the citywide analytical source.

In [5]:
from pathlib import Path

import pandas as pd

from dash import Dash, Input, Output, dcc, html

from dashboard.analysis_windows import (
    get_analysis_bounds,
    get_history_bounds,
    get_previous_period,
)

from dashboard.crime_classification import (
    CANONICAL_CRIME_TYPES,
)

from dashboard.crime_controls import (
    make_analysis_controls,
    make_analysis_state,
    make_neighborhood_options,
    validate_analysis_dates,
)

from dashboard.crime_filters import (
    filter_crime_records,
)

In [6]:
crime = crime_context["valid_time"].copy()

history_start, history_end = get_history_bounds(
    crime["offense_date"]
)

analysis_start, analysis_end = get_analysis_bounds(
    history_end
)

analysis_start = analysis_start.date().isoformat()
analysis_end = analysis_end.date().isoformat()

default_start = history_end.date().isoformat()
default_end = history_end.date().isoformat()

default_categories = list(CANONICAL_CRIME_TYPES)
crime_category_options = [
    {"label": category.title(), "value": category}
    for category in default_categories
]

crime_subcategory_options = [
    {"label": value.title(), "value": value}
    for value in (
        crime["offense_sub_category"]
        .dropna()
        .astype("string")
        .str.strip()
        .sort_values()
        .unique()
        .tolist()
    )
    if value not in ["", "nan", "none"]
]

crime_neighborhood_options = make_neighborhood_options(
    crime["mcpp_neighborhood"]
)

default_state = make_analysis_state(
    None, default_categories, [], [],
    default_start, default_end, default_categories,
)

In [7]:
CARD_BASE_STYLE = {
    "width": "350px",
    "minHeight": "150px",
    "padding": "16px 18px",

    "display": "flex",
    "justifyContent": "space-between",
    "gap": "24px",
    "alignItems": "center",

    "background": "#181818",
    "border": "1px solid #333333",
    "borderRadius": "8px",
    "boxSizing": "border-box",
    "transition": (
        "background-color 160ms ease, "
        "border-color 160ms ease"
    ),
}


POSITIVE_CARD_STYLE = {
    "background": "rgba(34, 197, 94, 0.08)",
    "border": "1px solid #22c55e",
}


NEGATIVE_CARD_STYLE = {
    "background": "rgba(249, 115, 22, 0.08)",
    "border": "1px solid #f97316",
}


NEUTRAL_CARD_STYLE = {
    "background": "#181818",
    "border": "1px solid #333333",
}


def get_change_presentation(raw_change):
    """
    Return the directional arrow, text color, and card treatment
    for a change relative to the previous analysis period.
    """
    if raw_change > 0:
        return {
            "arrow": "↗",
            "color": "#22c55e",
            "card_style": POSITIVE_CARD_STYLE,
        }

    if raw_change < 0:
        return {
            "arrow": "↘",
            "color": "#f97316",
            "card_style": NEGATIVE_CARD_STYLE,
        }

    return {
        "arrow": "→",
        "color": "#bbbbbb",
        "card_style": NEUTRAL_CARD_STYLE,
    }

In [8]:
from pathlib import Path


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


kpi_app = Dash(
    __name__,
    assets_folder=str(PROJECT_ROOT / "assets"),
)


kpi_app.layout = html.Div(
    children=[
        # -------------------------------------------------
        # PAGE TITLE
        # -------------------------------------------------
        html.Div(
            children=[
                html.H2(
                    "Crime KPI Prototype",
                    style={
                        "margin": "0",
                        "color": "white",
                        "fontSize": "19px",
                    },
                ),
            ],
            style={
                "padding": "10px 12px",
                "background": "#111111",
            },
        ),

        # -------------------------------------------------
        # PRODUCTION GLOBAL CRIME CONTROLS
        # -------------------------------------------------
        make_analysis_controls(
            default_state,
            crime_category_options,
            default_categories,
            crime_subcategory_options,
            crime_neighborhood_options,
            analysis_start,
            analysis_end,
        ),

        # -------------------------------------------------
        # KPI PREVIEW AREA
        # -------------------------------------------------
        html.Div(
            children=[
                html.Div(
                    id="prototype-crime-card",
                    children=[
                        # ---------------------------------
                        # LEFT SIDE
                        # ---------------------------------
                        html.Div(
                            children=[
                                html.Div(
                                    "Overall Crime Count",
                                    style={
                                        "color": "#bbbbbb",
                                        "fontSize": "11px",
                                        "fontWeight": "600",
                                        "letterSpacing": "0.04em",
                                        "textTransform": "uppercase",
                                    },
                                ),

                                html.Div(
                                    id="prototype-crime-count",
                                    style={
                                        "color": "#ffffff",
                                        "fontSize": "36px",
                                        "fontWeight": "700",
                                        "lineHeight": "40px",
                                        "marginTop": "3px",
                                    },
                                ),

                                html.Div(
                                    id="prototype-comparison-period",
                                    style={
                                        "color": "#888888",
                                        "fontSize": "10px",
                                        "lineHeight": "14px",
                                        "marginTop": "5px",
                                    },
                                ),
                            ],
                            style={
                                "minWidth": "0",
                                "flex": "1",
                            },
                        ),

                        # ---------------------------------
                        # RIGHT SIDE
                        # ---------------------------------
                        html.Div(
                            children=[
                                html.Div(
                                    id="prototype-change-value",
                                    style={
                                        "fontSize": "26px",
                                        "fontWeight": "700",
                                        "lineHeight": "30px",
                                        "whiteSpace": "nowrap",
                                        "textAlign": "right",
                                    },
                                ),

                                dcc.RadioItems(
                                    id="prototype-change-mode",
                                    options=[
                                        {
                                            "label": "Raw",
                                            "value": "raw",
                                        },
                                        {
                                            "label": "%",
                                            "value": "percent",
                                        },
                                    ],
                                    value="raw",
                                    inline=False, #Find
                                    inputStyle={
                                        "marginRight": "3px",
                                    },
                                    labelStyle={
                                        "marginLeft": "0px",
                                        "cursor": "pointer",
                                        "color": "#dddddd",
                                    },
                                    style={
                                        "width": "100%",
                                        "display": "flex",
                                        "justifyContent": "flex-end",
                                        "columnGap": "12px",
                                        "marginTop": "30px",
                                        "fontSize": "10px",
                                        "whiteSpace": "nowrap",
                                    },
                                ),
                            ],
                            style={
                                "display": "flex",
                                "flexDirection": "column",
                                "alignItems": "flex-end",
                                "justifyContent": "center",
                                "flexShrink": "0",
                            },
                        ),
                    ],
                    style=CARD_BASE_STYLE,
                ),
            ],
            style={
                "padding": "24px",
                "background": "#111111",
            },
        ),
    ],
    style={
        "background": "#111111",
        "minHeight": "100vh",
        "fontFamily": "Arial, sans-serif",
    },
)

In [9]:
@kpi_app.callback(
    Output(
        "prototype-crime-count",
        "children",
    ),
    Output(
        "prototype-change-value",
        "children",
    ),
    Output(
        "prototype-comparison-period",
        "children",
    ),
    Output(
        "prototype-crime-card",
        "style",
    ),
    Input(
        "crime-analysis-start-date-input",
        "value",
    ),
    Input(
        "crime-analysis-end-date-input",
        "value",
    ),
    Input(
        "crime-category-filter",
        "value",
    ),
    Input(
        "crime-subcategory-filter",
        "value",
    ),
    Input(
        "crime-neighborhood-filter",
        "value",
    ),
    Input(
        "prototype-change-mode",
        "value",
    ),
)
def update_crime_count_prototype(
    start_date,
    end_date,
    categories,
    subcategories,
    neighborhoods,
    change_mode,
):
    bounded_dates = validate_analysis_dates(
        start_date,
        end_date,
        analysis_start,
        analysis_end,
        clamp=True,
    )

    if bounded_dates is None:
        return (
            "—",
            "—",
            "Invalid analysis period",
            CARD_BASE_STYLE,
        )

    current_start, current_end = bounded_dates

    current_state = {
        "start_date": current_start,
        "end_date": current_end,
        "crime_categories": categories or [],
        "crime_subcategories": subcategories or [],
        "neighborhoods": neighborhoods or [],
    }

    # ----------------------------
    # CURRENT PERIOD
    # ----------------------------
    current_records = filter_crime_records(
        crime,
        current_state,
    )

    current_count = (
        current_records["offense_id"]
        .nunique()
    )

    # ----------------------------
    # PREVIOUS EQUAL-LENGTH PERIOD
    # ----------------------------
    previous_period = get_previous_period(
        current_start,
        current_end,
        history_bounds=(
            history_start,
            history_end,
        ),
    )

    if previous_period is None:
        return (
            f"{current_count:,}",
            "—",
            "Previous period unavailable",
            CARD_BASE_STYLE,
        )

    previous_start, previous_end = previous_period

    previous_state = {
        **current_state,
        "start_date": previous_start.date().isoformat(),
        "end_date": previous_end.date().isoformat(),
    }

    previous_records = filter_crime_records(
        crime,
        previous_state,
    )

    previous_count = (
        previous_records["offense_id"]
        .nunique()
    )

    # ----------------------------
    # CHANGE
    # ----------------------------
    raw_change = current_count - previous_count

    presentation = get_change_presentation(
        raw_change
    )

    arrow = presentation["arrow"]

    if change_mode == "percent":
        if previous_count == 0:
            change_number = "—"
        else:
            percent_change = (
                raw_change
                / previous_count
                * 100
            )

            change_number = (
                f"{abs(percent_change):.1f}%"
            )

    else:
        change_number = (
            f"{abs(raw_change):,}"
        )

    change_display = html.Span(
        children=[
            html.Span(
                arrow,
                style={
                    "fontSize": "24px",
                    "marginRight": "5px",
                },
            ),
            html.Span(
                change_number,
            ),
        ],
        style={
            "color": presentation["color"],
        },
    )

    # ----------------------------
    # COMPARISON LABEL
    # ----------------------------
    if previous_start == previous_end:
        previous_date_label = (
            previous_start.strftime(
                "%b %d, %Y"
            )
        )
    else:
        previous_date_label = (
            f"{previous_start.strftime('%b %d, %Y')}"
            " – "
            f"{previous_end.strftime('%b %d, %Y')}"
        )

    previous_label = (
        f"Previous: {previous_date_label}"
        f"  •  {previous_count:,} offenses"
    )

    # ----------------------------
    # CONDITIONAL CARD APPEARANCE
    # ----------------------------
    card_style = {
        **CARD_BASE_STYLE,
        **presentation["card_style"],
    }

    return (
        f"{current_count:,}",
        change_display,
        previous_label,
        card_style,
    )

In [10]:
kpi_app.run(
    jupyter_mode="inline",
    debug=False,
    port=8052,
)

## 4. Overall crime rate KPI (Assignee: Parfait)

**Numerator:** distinct `offense_id` from `crime_context["valid_time"]`

**Denominator:** `crime_context["city_population"]`

**Formula:** `count / city_population * 100_000`

Use the direct Seattle population value. Do **not** sum neighborhood populations and call that the city denominator.

In [11]:
#crime_count = selected_crime["offense_id"].nunique()
#city_population = crime_context["city_population"]

#overall_crime_rate_per_100k = crime_count / city_population * 100_000
#overall_crime_rate_per_100k

## 5. Top-level crime category counts (Assignee: Ben Carr)

**Authoritative source:** `crime_context["valid_time"]`

**Category field:** `offense_category` (also mirrored into `event_importance_bin` for dashboard compatibility)

The current canonical analytical categories are:

- `crimes against persons`
- `crimes against property`
- `crimes against society / other`

In [12]:
def get_category_counts(
    crime,
    state,
    categories=CANONICAL_CRIME_TYPES,
):
    counts = {}

    for category in categories:
        category_state = {
            **state,
            "crime_categories": [category],
        }

        selected = filter_crime_records(
            crime,
            category_state,
        )

        counts[category] = (
            selected["offense_id"]
            .nunique()
        )

    return counts

In [13]:
current_state = default_state.copy()

previous_period = get_previous_period(
    current_state["start_date"],
    current_state["end_date"],
    history_bounds=(
        history_start,
        history_end,
    ),
)

if previous_period is None:
    raise ValueError(
        "Previous comparison period is unavailable."
    )

previous_start, previous_end = previous_period

previous_state = {
    **current_state,
    "start_date": previous_start.date().isoformat(),
    "end_date": previous_end.date().isoformat(),
}

current_category_counts = get_category_counts(
    crime,
    current_state,
)

previous_category_counts = get_category_counts(
    crime,
    previous_state,
)

for category in CANONICAL_CRIME_TYPES:
    current = current_category_counts[category]
    previous = previous_category_counts[category]

    print(
        category,
        "current:",
        current,
        "previous:",
        previous,
        "change:",
        current - previous,
    )

crimes against persons current: 14 previous: 18 change: -4
crimes against property current: 10 previous: 38 change: -28
crimes against society / other current: 16 previous: 26 change: -10


In [14]:
CATEGORY_LABELS = {
    "crimes against persons": "Crimes Against Persons",
    "crimes against property": "Crimes Against Property",
    "crimes against society / other": "Crimes Against Society / Other",
}


def category_change_display(current, previous):
    raw_change = current - previous

    if raw_change > 0:
        return "↗", abs(raw_change), "#22c55e"

    if raw_change < 0:
        return "↘", abs(raw_change), "#f97316"

    return "→", 0, "#bbbbbb"

In [15]:
def build_category_table_body(
    current_counts,
    previous_counts,
    change_mode="raw",
):
    columns = []

    for category in CANONICAL_CRIME_TYPES:
        current = current_counts[category]
        previous = previous_counts[category]

        raw_change = current - previous

        arrow, _, change_color = category_change_display(
            current,
            previous,
        )

        if change_mode == "percent":
            if previous == 0:
                change_text = f"{arrow} —"
            else:
                percent_change = (
                    raw_change
                    / previous
                    * 100
                )

                change_text = (
                    f"{arrow} "
                    f"{abs(percent_change):.1f}%"
                )
        else:
            change_text = (
                f"{arrow} "
                f"{abs(raw_change):,}"
            )

        columns.append(
            html.Div(
                children=[
                    html.Div(
                        CATEGORY_LABELS.get(
                            category,
                            str(category).title(),
                        ),
                        style={
                            "color": "#bbbbbb",
                            "fontSize": "11px",
                            "fontWeight": "600",
                            "lineHeight": "14px",
                            "textAlign": "center",
                        },
                    ),

                    html.Div(
                        f"{current:,}",
                        style={
                            "color": "#ffffff",
                            "fontSize": "20px",
                            "fontWeight": "700",
                            "textAlign": "center",
                        },
                    ),

                    html.Div(
                        f"{previous:,}",
                        style={
                            "color": "#888888",
                            "fontSize": "14px",
                            "fontWeight": "500",
                            "textAlign": "center",
                        },
                    ),

                    html.Div(
                        change_text,
                        style={
                            "color": change_color,
                            "fontSize": "16px",
                            "fontWeight": "700",
                            "textAlign": "center",
                        },
                    ),
                ],
                style={
                    "display": "grid",
                    "gridTemplateRows": (
                        "44px 44px 44px 44px"
                    ),
                    "alignItems": "center",
                    "borderLeft": "1px solid #2d2d2d",
                    "padding": "0 12px",
                },
            )
        )

    return [
        # Row labels
        html.Div(
            children=[
                html.Div(""),
                html.Div("Current"),
                html.Div("Previous"),
                html.Div("Change"),
            ],
            style={
                "display": "grid",
                "gridTemplateRows": (
                    "44px 44px 44px 44px"
                ),
                "alignItems": "center",
                "color": "#888888",
                "fontSize": "11px",
                "fontWeight": "600",
            },
        ),

        *columns,
    ]

In [16]:
category_table = html.Div(
    children=[
        # -----------------------------------------------
        # HEADER
        # -----------------------------------------------
        html.Div(
            children=[
                html.Div(
                    "Top-Level Crime Categories",
                    style={
                        "color": "#bbbbbb",
                        "fontSize": "11px",
                        "fontWeight": "600",
                        "letterSpacing": "0.04em",
                        "textTransform": "uppercase",
                    },
                ),

                dcc.RadioItems(
                    id="prototype-category-change-mode",
                    options=[
                        {
                            "label": "Raw",
                            "value": "raw",
                        },
                        {
                            "label": "%",
                            "value": "percent",
                        },
                    ],
                    value="raw",
                    inline=True,
                    inputStyle={
                        "marginRight": "4px",
                    },
                    labelStyle={
                        "marginLeft": "10px",
                        "cursor": "pointer",
                        "color": "#dddddd",
                    },
                    style={
                        "fontSize": "10px",
                        "whiteSpace": "nowrap",
                    },
                ),
            ],
            style={
                "display": "flex",
                "alignItems": "center",
                "justifyContent": "space-between",
                "padding": "12px 16px",
                "borderBottom": "1px solid #333333",
            },
        ),

        # -----------------------------------------------
        # DYNAMIC TABLE BODY
        # -----------------------------------------------
        html.Div(
            id="prototype-category-table-body",
            style={
                "display": "grid",
                "gridTemplateColumns": (
                    "90px repeat(3, minmax(150px, 1fr))"
                ),
                "alignItems": "stretch",
                "padding": "4px 16px 12px",
            },
        ),

        # -----------------------------------------------
        # PREVIOUS PERIOD NOTE
        # -----------------------------------------------
        html.Div(
            id="prototype-category-period-note",
            style={
                "padding": "0 16px 12px",
                "color": "#666666",
                "fontSize": "10px",
            },
        ),
    ],
    style={
        "width": "100%",
        "maxWidth": "760px",
        "background": "#181818",
        "border": "1px solid #333333",
        "borderRadius": "8px",
        "overflow": "hidden",
        "boxSizing": "border-box",
    },
)

In [17]:
preview_app = Dash(
    __name__,
)

preview_app.layout = html.Div(
    children=[
        make_analysis_controls(
            default_state,
            crime_category_options,
            default_categories,
            crime_subcategory_options,
            crime_neighborhood_options,
            analysis_start,
            analysis_end,
        ),

        html.Div(
            children=category_table,
            style={
                "padding": "24px",
            },
        ),
    ],
    style={
        "background": "#111111",
        "minHeight": "100vh",
        "fontFamily": "Arial, sans-serif",
    },
)

In [18]:
@preview_app.callback(
    Output(
        "prototype-category-table-body",
        "children",
    ),
    Output(
        "prototype-category-period-note",
        "children",
    ),
    Input(
        "crime-analysis-start-date-input",
        "value",
    ),
    Input(
        "crime-analysis-end-date-input",
        "value",
    ),
    Input(
        "prototype-category-change-mode",
        "value",
    ),
)
def update_category_table(
    start_date,
    end_date,
    change_mode,
):
    bounded_dates = validate_analysis_dates(
        start_date,
        end_date,
        analysis_start,
        analysis_end,
        clamp=True,
    )

    if bounded_dates is None:
        return [], "Invalid analysis period"

    current_start, current_end = bounded_dates

    # Deliberately ignore all non-date global filters.
    current_state = {
        "start_date": current_start,
        "end_date": current_end,
        "crime_categories": [],
        "crime_subcategories": [],
        "neighborhoods": [],
    }

    current_counts = get_category_counts(
        crime,
        current_state,
    )

    previous_period = get_previous_period(
        current_start,
        current_end,
        history_bounds=(
            history_start,
            history_end,
        ),
    )

    if previous_period is None:
        return [], "Previous period unavailable"

    previous_start, previous_end = previous_period

    previous_state = {
        **current_state,
        "start_date": (
            previous_start.date().isoformat()
        ),
        "end_date": (
            previous_end.date().isoformat()
        ),
    }

    previous_counts = get_category_counts(
        crime,
        previous_state,
    )

    body = build_category_table_body(
        current_counts,
        previous_counts,
        change_mode=change_mode,
    )

    if previous_start == previous_end:
        previous_period_label = (
            previous_start.strftime("%b %d, %Y")
        )
    else:
        previous_period_label = (
            f"{previous_start.strftime('%b %d, %Y')}"
            " – "
            f"{previous_end.strftime('%b %d, %Y')}"
        )

    note = (
        f"Previous period: {previous_period_label}"
    )

    return body, note

In [19]:
preview_app.run(
    jupyter_mode="inline",
    debug=False,
    port=8053,
)

## 6. Median qualified response-time KPI (Assignee: Ben Carr)

**Authoritative source:** `calls_context["response_analysis"]`

**Grain:** one row per `cad_event_number`

**Date field:** `queued_time`

**Statistic:** median `response_time_minutes`

Do **not** derive this KPI from the calls map hover data or from raw dispatch rows. `response_analysis` is the production event-level qualified-response table.

In [20]:
response = calls_context["response_analysis"].copy()

response["queued_time"] = pd.to_datetime(
    response["queued_time"],
    errors="coerce",
)

response_start = (
    response["queued_time"]
    .max()
    .normalize()
)

response_end = response_start

response_selected = response.loc[
    response["queued_time"]
    .dt.normalize()
    .between(
        response_start,
        response_end,
    )
].copy()

response_selected[
    [
        "queued_time",
        "priority",
        "response_time_minutes",
    ]
].head()

,queued_time,priority,response_time_minutes
608711,2026-09-11 00:05:00,2,25.566667
608712,2026-09-11 00:05:44,3,118.450000
608713,2026-09-11 00:07:19,7,0.000000
608714,2026-09-11 00:10:34,3,0.016667
608715,2026-09-11 00:15:55,3,11.033333


In [21]:
priority_tests = {
    "Priority 1–3": [1, 2, 3],
    "Priority 1–2": [1, 2],
    "Priority 1 only": [1],
}

priority_comparison = []

for label, priorities in priority_tests.items():
    selected = response_selected.loc[
        response_selected["priority"].isin(
            priorities
        )
    ].copy()

    priority_comparison.append(
        {
            "qualification": label,
            "priorities": priorities,
            "events": len(selected),
            "median_response_minutes": (
                selected[
                    "response_time_minutes"
                ].median()
            ),
        }
    )

priority_comparison = pd.DataFrame(
    priority_comparison
)

priority_comparison

,qualification,priorities,events,median_response_minutes
0,Priority 1–3,"[1, 2, 3]",589,16.900000
1,Priority 1–2,"[1, 2]",369,14.650000
2,Priority 1 only,[1],101,6.516667


In [22]:
priority_comparison.assign(
    median_response_minutes=lambda df: (
        df["median_response_minutes"]
        .round(1)
    )
)

,qualification,priorities,events,median_response_minutes
0,Priority 1–3,"[1, 2, 3]",589,16.9
1,Priority 1–2,"[1, 2]",369,14.6
2,Priority 1 only,[1],101,6.5


In [23]:
RESPONSE_PRIORITY_OPTIONS = {
    "Priority 1–3": [1, 2, 3],
    "Priority 1–2": [1, 2],
    "Priority 1 only": [1],
}


def get_response_kpi_values(
    response,
    start_date,
    end_date,
    priorities,
    history_start,
    history_end,
):
    response = response.copy()
    response["queued_time"] = pd.to_datetime(
        response["queued_time"],
        errors="coerce",
    )

    current_selected = response.loc[
        response["queued_time"]
        .dt.normalize()
        .between(
            pd.to_datetime(start_date),
            pd.to_datetime(end_date),
        )
        & response["priority"].isin(priorities)
    ].copy()

    current_median = current_selected[
        "response_time_minutes"
    ].median()

    current_events = len(current_selected)

    previous_period = get_previous_period(
        start_date,
        end_date,
        history_bounds=(
            history_start,
            history_end,
        ),
    )

    if previous_period is None:
        return {
            "current_median": current_median,
            "current_events": current_events,
            "previous_median": np.nan,
            "previous_events": 0,
            "previous_start": None,
            "previous_end": None,
        }

    previous_start, previous_end = previous_period

    previous_selected = response.loc[
        response["queued_time"]
        .dt.normalize()
        .between(
            previous_start,
            previous_end,
        )
        & response["priority"].isin(priorities)
    ].copy()

    previous_median = previous_selected[
        "response_time_minutes"
    ].median()

    previous_events = len(previous_selected)

    return {
        "current_median": current_median,
        "current_events": current_events,
        "previous_median": previous_median,
        "previous_events": previous_events,
        "previous_start": previous_start,
        "previous_end": previous_end,
    }


def get_response_change_presentation(
    current_median,
    previous_median,
):
    if (
        pd.isna(current_median)
        or pd.isna(previous_median)
    ):
        return {
            "arrow": "→",
            "color": "#bbbbbb",
            "card_style": {
                **CARD_BASE_STYLE,
                **NEUTRAL_CARD_STYLE,
            },
            "raw_change": np.nan,
        }

    raw_change = current_median - previous_median

    # For response time:
    # lower is better, higher is worse.
    if raw_change < 0:
        return {
            "arrow": "↘",
            "color": "#22c55e",
            "card_style": {
                **CARD_BASE_STYLE,
                **POSITIVE_CARD_STYLE,
            },
            "raw_change": raw_change,
        }

    if raw_change > 0:
        return {
            "arrow": "↗",
            "color": "#f97316",
            "card_style": {
                **CARD_BASE_STYLE,
                **NEGATIVE_CARD_STYLE,
            },
            "raw_change": raw_change,
        }

    return {
        "arrow": "→",
        "color": "#bbbbbb",
        "card_style": {
            **CARD_BASE_STYLE,
            **NEUTRAL_CARD_STYLE,
        },
        "raw_change": raw_change,
    }

In [24]:
response = calls_context["response_analysis"].copy()

response["queued_time"] = pd.to_datetime(
    response["queued_time"],
    errors="coerce",
)

response_history_start = (
    response["queued_time"]
    .min()
    .normalize()
)
response_history_end = (
    response["queued_time"]
    .max()
    .normalize()
)

response_analysis_start, response_analysis_end = (
    get_analysis_bounds(response_history_end)
)

response_default_state = {
    "start_date": str(response_analysis_end.date()),
    "end_date": str(response_analysis_end.date()),
    "crime_categories": [],
    "crime_subcategories": [],
    "neighborhoods": [],
}


response_kpi_app = Dash(__name__)

response_kpi_app.layout = html.Div(
    children=[
        html.Div(
            children=[
                html.H2(
                    "Response-Time KPI Prototype",
                    style={
                        "margin": "0",
                        "color": "white",
                        "fontSize": "19px",
                    },
                ),
            ],
            style={
                "padding": "10px 12px",
                "background": "#111111",
            },
        ),

        make_analysis_controls(
            response_default_state,
            crime_category_options,
            default_categories,
            crime_subcategory_options,
            crime_neighborhood_options,
            response_analysis_start,
            response_analysis_end,
        ),

        html.Div(
            children=[
                html.Div(
                    children=[
                        html.Div(
                            "Qualified Priority Filter",
                            style={
                                "color": "#bbbbbb",
                                "fontSize": "11px",
                                "fontWeight": "600",
                                "letterSpacing": "0.04em",
                                "textTransform": "uppercase",
                                "marginBottom": "8px",
                            },
                        ),
                        dcc.RadioItems(
                            id="prototype-response-priority-scope",
                            options=[
                                {
                                    "label": label,
                                    "value": label,
                                }
                                for label in RESPONSE_PRIORITY_OPTIONS
                            ],
                            value="Priority 1–3",
                            inline=True,
                            inputStyle={
                                "marginRight": "4px",
                            },
                            labelStyle={
                                "marginRight": "16px",
                                "cursor": "pointer",
                                "color": "#dddddd",
                            },
                            style={
                                "fontSize": "12px",
                                "whiteSpace": "nowrap",
                            },
                        ),
                    ],
                    style={
                        "marginBottom": "18px",
                    },
                ),

                html.Div(
                    id="prototype-response-card",
                    children=[
                        # LEFT SIDE
                        html.Div(
                            children=[
                                html.Div(
                                    "Median Qualified Response Time",
                                    style={
                                        "color": "#bbbbbb",
                                        "fontSize": "11px",
                                        "fontWeight": "600",
                                        "letterSpacing": "0.04em",
                                        "textTransform": "uppercase",
                                    },
                                ),
                                html.Div(
                                    id="prototype-response-median",
                                    style={
                                        "color": "#ffffff",
                                        "fontSize": "36px",
                                        "fontWeight": "700",
                                        "lineHeight": "40px",
                                        "marginTop": "3px",
                                    },
                                ),
                                html.Div(
                                    id="prototype-response-comparison-period",
                                    style={
                                        "color": "#888888",
                                        "fontSize": "10px",
                                        "lineHeight": "14px",
                                        "marginTop": "5px",
                                    },
                                ),
                            ],
                            style={
                                "minWidth": "0",
                                "flex": "1",
                            },
                        ),

                        # RIGHT SIDE
                        html.Div(
                            children=[
                                html.Div(
                                    id="prototype-response-change-value",
                                    style={
                                        "fontSize": "26px",
                                        "fontWeight": "700",
                                        "lineHeight": "30px",
                                        "whiteSpace": "nowrap",
                                        "textAlign": "right",
                                    },
                                ),
                                dcc.RadioItems(
                                    id="prototype-response-change-mode",
                                    options=[
                                        {
                                            "label": "Raw",
                                            "value": "raw",
                                        },
                                        {
                                            "label": "%",
                                            "value": "percent",
                                        },
                                    ],
                                    value="raw",
                                    inline=True,
                                    inputStyle={
                                        "marginRight": "3px",
                                    },
                                    labelStyle={
                                        "marginLeft": "8px",
                                        "cursor": "pointer",
                                        "color": "#dddddd",
                                    },
                                    style={
                                        "marginTop": "30px",
                                        "fontSize": "10px",
                                        "whiteSpace": "nowrap",
                                    },
                                ),
                            ],
                            style={
                                "display": "flex",
                                "flexDirection": "column",
                                "alignItems": "flex-end",
                                "justifyContent": "center",
                                "flexShrink": "0",
                            },
                        ),
                    ],
                    style=CARD_BASE_STYLE,
                ),
            ],
            style={
                "padding": "24px",
                "background": "#111111",
            },
        ),
    ],
    style={
        "background": "#111111",
        "minHeight": "100vh",
        "fontFamily": "Arial, sans-serif",
    },
)

In [25]:
@response_kpi_app.callback(
    Output(
        "prototype-response-median",
        "children",
    ),
    Output(
        "prototype-response-change-value",
        "children",
    ),
    Output(
        "prototype-response-comparison-period",
        "children",
    ),
    Output(
        "prototype-response-card",
        "style",
    ),
    Input(
        "crime-analysis-start-date-input",
        "value",
    ),
    Input(
        "crime-analysis-end-date-input",
        "value",
    ),
    Input(
        "prototype-response-priority-scope",
        "value",
    ),
    Input(
        "prototype-response-change-mode",
        "value",
    ),
)
def update_response_kpi(
    start_date,
    end_date,
    priority_scope_label,
    change_mode,
):
    bounded_dates = validate_analysis_dates(
        start_date,
        end_date,
        response_analysis_start,
        response_analysis_end,
        clamp=True,
    )

    if bounded_dates is None:
        return (
            "—",
            "→ —",
            "Invalid analysis period",
            {
                **CARD_BASE_STYLE,
                **NEUTRAL_CARD_STYLE,
            },
        )

    current_start, current_end = bounded_dates
    priorities = RESPONSE_PRIORITY_OPTIONS[
        priority_scope_label
    ]

    values = get_response_kpi_values(
        response=response,
        start_date=current_start,
        end_date=current_end,
        priorities=priorities,
        history_start=response_history_start,
        history_end=response_history_end,
    )

    current_median = values["current_median"]
    previous_median = values["previous_median"]
    previous_start = values["previous_start"]
    previous_end = values["previous_end"]

    if pd.isna(current_median):
        current_text = "—"
    else:
        current_text = f"{current_median:.1f} min"

    presentation = get_response_change_presentation(
        current_median,
        previous_median,
    )

    raw_change = presentation["raw_change"]
    arrow = presentation["arrow"]
    color = presentation["color"]
    card_style = presentation["card_style"]

    if pd.isna(raw_change):
        change_text = f"{arrow} —"
    elif change_mode == "percent":
        if (
            pd.isna(previous_median)
            or previous_median == 0
        ):
            change_text = f"{arrow} —"
        else:
            pct_change = (
                raw_change
                / previous_median
                * 100
            )
            change_text = (
                f"{arrow} "
                f"{abs(pct_change):.1f}%"
            )
    else:
        change_text = (
            f"{arrow} "
            f"{abs(raw_change):.1f} min"
        )

    change_component = html.Div(
        change_text,
        style={
            "color": color,
        },
    )

    if previous_start is None or previous_end is None:
        comparison_text = (
            "Previous period unavailable"
        )
    else:
        if previous_start == previous_end:
            previous_label = previous_start.strftime(
                "%b %d, %Y"
            )
        else:
            previous_label = (
                f"{previous_start.strftime('%b %d, %Y')}"
                " – "
                f"{previous_end.strftime('%b %d, %Y')}"
            )

        if pd.isna(previous_median):
            previous_value_text = "—"
        else:
            previous_value_text = (
                f"{previous_median:.1f} min"
            )

        comparison_text = (
            f"Previous: {previous_label}"
            f" • {previous_value_text}"
        )

    return (
        current_text,
        change_component,
        comparison_text,
        card_style,
    )

In [26]:
response_kpi_app.run(
    jupyter_mode="inline",
    debug=False,
    port=8054,
)

## 7. UOF incident KPI (Assignee: Parfait)

**Authoritative source:** `uof_context["df"]`

**Identifier:** distinct `incident_num`

**Date field:** `occured_date_time` (source spelling retained)

Use the production helper `count_uof_incidents()` rather than counting rows.

In [27]:
from dashboard.uof_dashboard_data import count_uof_incidents

uof_latest = uof_context["latest_available_date"]

if pd.notna(uof_latest):
    uof_day = str(uof_latest.date())
    uof_incident_count = count_uof_incidents(
        uof_context["df"],
        uof_day,
        uof_day,
    )
    print("UOF incidents:", uof_incident_count)

UOF incidents: 1


## 8. OIS event KPI (Assignee: Parfait)

**Authoritative source:** `uof_context["ois_events"]`

**Identifier:** distinct `ois_event_key`

**Event definition:** Seattle-local calendar day + normalized beat

Use `count_ois_events()` rather than counting UOF rows or distinct force incident numbers.

In [28]:
from dashboard.uof_dashboard_data import count_ois_events

if pd.notna(uof_latest):
    ois_day = str(uof_latest.date())
    ois_event_count = count_ois_events(
        uof_context["ois_events"],
        ois_day,
        ois_day,
    )
    print("Derived OIS events:", ois_event_count)

Derived OIS events: 0


## 9. Neighborhood crime ranking (Assignee: Parfait)

**Authoritative numerator source:** `crime_context["valid_time"]`

**Grouping field:** analytical `mcpp_neighborhood`

The analytical neighborhood field in `valid_time` uses spatial MCPP assignment when available and source neighborhood fallback otherwise. This is why rankings should be built from `valid_time`, not only from coordinate-valid map events.

For population-adjusted comparative rankings, join to `crime_context["neighborhood_population"]` and use the calibrated `population` field.

The current v1.1 methodology requires a `population >= 5_000` threshold for comparative rate ranking.

In [29]:
#neighborhood_counts = (
#    selected_crime
#    .dropna(subset=["mcpp_neighborhood"])
#    .groupby("mcpp_neighborhood")["offense_id"]
#    .nunique()
#    .rename("offense_count")
#    .reset_index()
#)

#neighborhood_rates = neighborhood_counts.merge(
#    crime_context["neighborhood_population"][
#        ["mcpp_neighborhood", "population"]
#    ],
#    on="mcpp_neighborhood",
#    how="left",
#)

#neighborhood_rates["crime_rate_per_100k"] = (
#    neighborhood_rates["offense_count"]
#    / neighborhood_rates["population"]
#    * 100_000
#)

#comparative_rate_ranking = (
#    neighborhood_rates.loc[neighborhood_rates["population"] >= 5_000]
#    .sort_values("crime_rate_per_100k", ascending=False)
#)

#comparative_rate_ranking.head()

## 10. Neighborhood response-time ranking (Assignee: Ben Carr)

**Authoritative source:** `calls_context["response_analysis"]`

**Grouping field:** `dispatch_neighborhood`

**Statistic:** median `response_time_minutes`

This ranking should be calculated from qualified event-level records. Coordinates are not required.

> The minimum qualified-event sample size and tie policy are still methodology decisions for the ranking component. Do not silently reuse the current scatter plot's 100-event threshold as a ranking rule unless the methodology is explicitly updated.

In [30]:
response_ranking_source = response_selected.dropna(
    subset=["dispatch_neighborhood"]
).copy()

response_ranking = (
    response_ranking_source
    .groupby("dispatch_neighborhood")
    .agg(
        qualified_events=("cad_event_number", "nunique"),
        median_response_minutes=("response_time_minutes", "median"),
    )
    .reset_index()
    .sort_values("median_response_minutes")
)

response_ranking.tail()

,dispatch_neighborhood,qualified_events,median_response_minutes
34,montlake/portage bay,3,99.416667
56,university,26,109.675000
25,hillman city,4,148.758333
29,madison park,1,298.900000
16,eastlake - west,2,306.991667


In [31]:
def build_neighborhood_response_ranking(
    response,
    start_date,
    end_date,
    priorities,
    min_events=1,
    top_n=10,
):
    selected = response.loc[
        response["queued_time"]
        .dt.normalize()
        .between(
            pd.to_datetime(start_date),
            pd.to_datetime(end_date),
        )
        & response["priority"].isin(priorities)
        & response["dispatch_neighborhood"].notna()
    ].copy()

    ranking = (
        selected
        .groupby(
            "dispatch_neighborhood",
            as_index=False,
        )
        .agg(
            median_response_minutes=(
                "response_time_minutes",
                "median",
            ),
            qualified_events=(
                "response_time_minutes",
                "size",
            ),
        )
    )

    ranking = ranking.loc[
        ranking["qualified_events"] >= min_events
    ].copy()

    ranking = (
        ranking
        .sort_values(
            [
                "median_response_minutes",
                "qualified_events",
                "dispatch_neighborhood",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    ranking["rank"] = (
        ranking.index + 1
    )

    return ranking

In [32]:
for label, priorities in RESPONSE_PRIORITY_OPTIONS.items():
    ranking = build_neighborhood_response_ranking(
        response=response,
        start_date=response_analysis_end,
        end_date=response_analysis_end,
        priorities=priorities,
        min_events=1,
        top_n=10,
    )

    print(f"\n{label}")
    display(
        ranking[
            [
                "rank",
                "dispatch_neighborhood",
                "median_response_minutes",
                "qualified_events",
            ]
        ]
    )


Priority 1–3


,rank,dispatch_neighborhood,median_response_minutes,qualified_events
0,1,madison park,298.900000,1
1,2,hillman city,223.233333,2
2,3,eastlake - east,157.533333,1
3,4,south park,105.933333,1
4,5,university,103.291667,16
5,6,phinney ridge,92.533333,5
6,7,montlake/portage bay,86.075000,2
7,8,greenwood,68.933333,10
8,9,madrona/leschi,61.191667,4
9,10,wallingford,54.408333,4



Priority 1–2


,rank,dispatch_neighborhood,median_response_minutes,qualified_events
0,1,madison park,298.900000,1
1,2,hillman city,223.066667,1
2,3,university,121.600000,6
3,4,montlake/portage bay,99.416667,1
4,5,phinney ridge,96.333333,3
5,6,greenwood,56.341667,6
6,7,belltown,45.516667,6
7,8,rainier beach,43.033333,6
8,9,ballard north,40.125000,4
9,10,rainier view,37.408333,2



Priority 1 only


,rank,dispatch_neighborhood,median_response_minutes,qualified_events
0,1,genesee,39.233333,1
1,2,madrona/leschi,18.500000,1
2,3,bitterlake,17.008333,2
3,4,claremont/rainier vista,13.783333,3
4,5,sandpoint,13.233333,3
5,6,ballard north,13.200000,1
6,7,north delridge,13.183333,1
7,8,fremont,11.891667,2
8,9,lakecity,10.416667,5
9,10,north admiral,10.183333,1


In [33]:
def build_response_ranking_rows(ranking):
    if ranking.empty:
        return [
            html.Div(
                "No qualified response events for this period.",
                style={
                    "padding": "18px",
                    "color": "#888888",
                    "fontSize": "12px",
                },
            )
        ]

    rows = []

    for row in ranking.itertuples():
        rows.append(
            html.Div(
                children=[
                    html.Div(
                        str(row.rank),
                        style={
                            "color": "#888888",
                            "fontSize": "12px",
                            "fontWeight": "600",
                        },
                    ),

                    html.Div(
                        str(
                            row.dispatch_neighborhood
                        ).title(),
                        style={
                            "color": "#ffffff",
                            "fontSize": "12px",
                            "fontWeight": "600",
                        },
                    ),

                    html.Div(
                        (
                            f"{row.median_response_minutes:.1f}"
                            " min"
                        ),
                        style={
                            "color": "#ffffff",
                            "fontSize": "14px",
                            "fontWeight": "700",
                            "textAlign": "right",
                        },
                    ),

                    html.Div(
                        f"{row.qualified_events:,}",
                        style={
                            "color": "#888888",
                            "fontSize": "12px",
                            "textAlign": "right",
                        },
                    ),
                ],
                style={
                    "display": "grid",
                    "gridTemplateColumns": (
                        "45px minmax(180px, 1fr) "
                        "120px 110px"
                    ),
                    "alignItems": "center",
                    "padding": "9px 14px",
                    "borderTop": (
                        "1px solid #2d2d2d"
                    ),
                },
            )
        )

    return rows

In [34]:
response_ranking_table = html.Div(
    children=[
        # -------------------------------------------------
        # HEADER / LOCAL CONTROLS
        # -------------------------------------------------
        html.Div(
            children=[
                html.Div(
                    "Neighborhood Response-Time Ranking",
                    style={
                        "color": "#bbbbbb",
                        "fontSize": "11px",
                        "fontWeight": "600",
                        "letterSpacing": "0.04em",
                        "textTransform": "uppercase",
                    },
                ),

                html.Div(
                    children=[
                        # Priority qualification
                        dcc.RadioItems(
                            id=(
                                "prototype-response-ranking-"
                                "priority-scope"
                            ),
                            options=[
                                {
                                    "label": label,
                                    "value": label,
                                }
                                for label
                                in RESPONSE_PRIORITY_OPTIONS
                            ],
                            value="Priority 1–3",
                            inline=True,
                            inputStyle={
                                "marginRight": "4px",
                            },
                            labelStyle={
                                "marginLeft": "12px",
                                "cursor": "pointer",
                                "color": "#dddddd",
                            },
                            style={
                                "fontSize": "10px",
                                "whiteSpace": "nowrap",
                            },
                        ),

                        # Minimum sample threshold
                        html.Div(
                            children=[
                                html.Span(
                                    "Min events",
                                    style={
                                        "color": "#888888",
                                        "fontSize": "10px",
                                        "marginRight": "6px",
                                    },
                                ),

                                dcc.Input(
                                    id=(
                                        "prototype-response-ranking-"
                                        "min-events"
                                    ),
                                    type="number",
                                    min=1,
                                    step=1,
                                    value=5,
                                    debounce=True,
                                    style={
                                        "width": "55px",
                                        "background": "#111111",
                                        "color": "#dddddd",
                                        "border": (
                                            "1px solid #444444"
                                        ),
                                        "borderRadius": "4px",
                                        "padding": "3px 5px",
                                        "fontSize": "10px",
                                    },
                                ),
                            ],
                            style={
                                "display": "flex",
                                "alignItems": "center",
                                "marginLeft": "18px",
                            },
                        ),
                    ],
                    style={
                        "display": "flex",
                        "alignItems": "center",
                    },
                ),
            ],
            style={
                "display": "flex",
                "alignItems": "center",
                "justifyContent": "space-between",
                "padding": "12px 14px",
            },
        ),

        # -------------------------------------------------
        # COLUMN HEADERS
        # -------------------------------------------------
        html.Div(
            children=[
                html.Div("Rank"),
                html.Div("Neighborhood"),
                html.Div(
                    "Median Response",
                    style={
                        "textAlign": "right",
                    },
                ),
                html.Div(
                    "Qualified Events",
                    style={
                        "textAlign": "right",
                    },
                ),
            ],
            style={
                "display": "grid",
                "gridTemplateColumns": (
                    "45px minmax(180px, 1fr) "
                    "120px 110px"
                ),
                "padding": "8px 14px",
                "borderTop": "1px solid #333333",
                "color": "#777777",
                "fontSize": "10px",
                "fontWeight": "600",
                "textTransform": "uppercase",
            },
        ),

        # -------------------------------------------------
        # DYNAMIC ROWS
        # -------------------------------------------------
        html.Div(
            id="prototype-response-ranking-body",
        ),
    ],
    style={
        "width": "100%",
        "maxWidth": "760px",
        "background": "#181818",
        "border": "1px solid #333333",
        "borderRadius": "8px",
        "overflow": "hidden",
        "boxSizing": "border-box",
    },
)

In [35]:
response_ranking_app = Dash(__name__)

response_ranking_app.layout = html.Div(
    children=[
        make_analysis_controls(
            response_default_state,
            crime_category_options,
            default_categories,
            crime_subcategory_options,
            crime_neighborhood_options,
            response_analysis_start,
            response_analysis_end,
        ),

        html.Div(
            response_ranking_table,
            style={
                "padding": "24px",
            },
        ),
    ],
    style={
        "background": "#111111",
        "minHeight": "100vh",
        "fontFamily": "Arial, sans-serif",
    },
)

In [36]:
@response_ranking_app.callback(
    Output(
        "prototype-response-ranking-body",
        "children",
    ),
    Input(
        "crime-analysis-start-date-input",
        "value",
    ),
    Input(
        "crime-analysis-end-date-input",
        "value",
    ),
    Input(
        "prototype-response-ranking-priority-scope",
        "value",
    ),
    Input(
        "prototype-response-ranking-min-events",
        "value",
    ),
)
def update_response_ranking(
    start_date,
    end_date,
    priority_scope_label,
    min_events,
):
    bounded_dates = validate_analysis_dates(
        start_date,
        end_date,
        response_analysis_start,
        response_analysis_end,
        clamp=True,
    )

    if bounded_dates is None:
        return [
            html.Div(
                "Invalid analysis period",
                style={
                    "padding": "18px",
                    "color": "#888888",
                },
            )
        ]

    current_start, current_end = bounded_dates

    priorities = RESPONSE_PRIORITY_OPTIONS[
        priority_scope_label
    ]

    # Keep the exploratory threshold valid.
    if min_events is None:
        min_events = 1

    min_events = max(
        1,
        int(min_events),
    )

    ranking = build_neighborhood_response_ranking(
        response=response,
        start_date=current_start,
        end_date=current_end,
        priorities=priorities,
        min_events=min_events,
        top_n=10,
    )

    return build_response_ranking_rows(
        ranking
    )

In [37]:
response_ranking_app.run(
    jupyter_mode="inline",
    debug=False,
    port=8055,
)

## 11. Crime choropleth (already exists/doesn't need significant revision)

For v1.1 analytical neighborhood shading, use:

- offense totals from `crime_context["valid_time"]`
- polygons from `crime_context["mcpp_boundaries"]`
- calibrated population from `crime_context["neighborhood_population"]`

The existing production map function currently has historical implementation differences, but the v1.1 analytical contract is to reconcile the choropleth with `valid_time` so coordinate-valid offenses without a spatial match can still use analytical neighborhood fallback.

For rates, use **per 100,000**, not per 1,000.

In [38]:
#choropleth_data = (
#    selected_crime
#    .dropna(subset=["mcpp_neighborhood"])
#    .groupby("mcpp_neighborhood")["offense_id"]
#    .nunique()
#    .rename("offense_count")
#    .reset_index()
#    .merge(
#        crime_context["neighborhood_population"][
#            ["mcpp_neighborhood", "population"]
#        ],
#        on="mcpp_neighborhood",
#        how="left",
#    )
#)

#choropleth_data["crime_rate_per_100k"] = (
#    choropleth_data["offense_count"]
#    / choropleth_data["population"]
#    * 100_000
#)

#choropleth_data.tail()

## 12. Crime point map (already exists/doesn't need significant revision)

**Authoritative point source:** `crime_context["event_mcpp"]`

Only coordinate-valid offenses can become points. The point layer is therefore intentionally narrower than the analytical population used for KPIs/rankings.

Use `crime_context["mcpp_boundaries"]` for polygon geometry.

Point-only filters such as free-text search should not change analytical neighborhood totals or choropleth shading.

In [ ]:
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from dash import Dash, Input, Output, State, dcc, html
from dash.exceptions import PreventUpdate

from dashboard.crime_classification import CANONICAL_CRIME_TYPES
from dashboard.crime_dashboard_data import (
    EVENT_ID_COLUMN,
    ROW_ID_COLUMN,
    TIME_COLUMN,
    REPORT_TIME_COLUMN,
    LAT_COL,
    LON_COL,
    CATEGORY_COLUMN,
    SUB_CATEGORY_COLUMN,
    normalize_neighborhood_name,
)
from dashboard.crime_dashboard_figures import CRIME_CATEGORY_COLOR_MAP
from dashboard.spd_config import (
    PLOTLY_MAP_STYLE,
    PLOTLY_SEATTLE_CENTER,
)


RATE_MIN_POPULATION = 5_000


def prepare_v11_point_source(context):
    """
    Keep event_mcpp as the authoritative point source, but attach the
    canonical analytical neighborhood assignment from valid_time.

    This lets global neighborhood filtering use the same neighborhood
    definition as KPIs, rankings, and the choropleth.
    """
    analytical = context["valid_time"].copy()
    points = context["event_mcpp"].copy()

    analytical_assignments = analytical[
        [EVENT_ID_COLUMN, "mcpp_neighborhood"]
    ].copy()

    analytical_assignments["mcpp_neighborhood"] = (
        normalize_neighborhood_name(
            analytical_assignments["mcpp_neighborhood"]
        )
    )

    analytical_assignments = analytical_assignments[
        analytical_assignments[EVENT_ID_COLUMN].notna()
        & analytical_assignments["mcpp_neighborhood"].notna()
        & (analytical_assignments["mcpp_neighborhood"] != "")
    ].drop_duplicates()

    # An offense should never resolve to multiple analytical MCPPs.
    assignment_counts = (
        analytical_assignments
        .groupby(EVENT_ID_COLUMN)["mcpp_neighborhood"]
        .nunique()
    )

    conflicting_ids = assignment_counts[
        assignment_counts > 1
    ]

    if not conflicting_ids.empty:
        raise ValueError(
            "Some offense_ids resolve to multiple analytical "
            "MCPP neighborhoods. Investigate before mapping."
        )

    analytical_lookup = (
        analytical_assignments
        .drop_duplicates(EVENT_ID_COLUMN)
        .rename(
            columns={
                "mcpp_neighborhood":
                    "analytical_mcpp_neighborhood"
            }
        )
    )

    points["spatial_mcpp_neighborhood"] = (
        normalize_neighborhood_name(
            points["mcpp_neighborhood"]
        )
    )

    points = (
        points
        .drop(
            columns=["analytical_mcpp_neighborhood"],
            errors="ignore",
        )
        .merge(
            analytical_lookup,
            on=EVENT_ID_COLUMN,
            how="left",
        )
    )

    # Global filtering should use the analytical assignment.
    # Spatial remains available separately for QA.
    points["mcpp_neighborhood"] = (
        points["analytical_mcpp_neighborhood"]
        .fillna(points["spatial_mcpp_neighborhood"])
    )

    return points


def prepare_v11_choropleth_data(
    context,
    selected_crime,
):
    """
    Build neighborhood metrics directly from the authoritative
    analytical crime population.
    """
    records = selected_crime.copy()

    records["mcpp_neighborhood"] = (
        normalize_neighborhood_name(
            records["mcpp_neighborhood"]
        )
    )

    boundaries = context["mcpp_boundaries"].copy()
    
    boundaries["mcpp_neighborhood"] = (
        normalize_neighborhood_name(
            boundaries["mcpp_neighborhood"]
        )
    )

    assignments = records[
        [EVENT_ID_COLUMN, "mcpp_neighborhood"]
    ].copy()

    assignments = assignments[
        assignments[EVENT_ID_COLUMN].notna()
        & assignments["mcpp_neighborhood"].notna()
        & assignments["mcpp_neighborhood"].isin(
            valid_mcpp_names
        )
    ].drop_duplicates()

    # Protect against accidentally counting one offense in multiple MCPPs.
    assignment_counts = (
        assignments
        .groupby(EVENT_ID_COLUMN)["mcpp_neighborhood"]
        .nunique()
    )

    conflicting_ids = assignment_counts[
        assignment_counts > 1
    ]

    if not conflicting_ids.empty:
        raise ValueError(
            "Selected crime contains offense_ids assigned "
            "to multiple MCPP neighborhoods."
        )

    neighborhood_counts = (
        assignments
        .groupby("mcpp_neighborhood")
        .agg(
            offense_count=(
                EVENT_ID_COLUMN,
                "nunique",
            )
        )
        .reset_index()
    )

    valid_mcpp_names = set(
        boundaries["mcpp_neighborhood"]
        .dropna()
        .astype(str)
    )

    population = (
        context["neighborhood_population"]
        .copy()
    )

    if "mcpp_neighborhood" not in population.columns:
        population = population.rename(
            columns={
                "dispatch_neighborhood":
                    "mcpp_neighborhood"
            }
        )

    population["mcpp_neighborhood"] = (
        normalize_neighborhood_name(
            population["mcpp_neighborhood"]
        )
    )

    population["population"] = pd.to_numeric(
        population["population"],
        errors="coerce",
    )

    population = (
        population[
            ["mcpp_neighborhood", "population"]
        ]
        .drop_duplicates("mcpp_neighborhood")
    )

    choropleth = (
        boundaries
        .merge(
            neighborhood_counts,
            on="mcpp_neighborhood",
            how="left",
        )
        .merge(
            population,
            on="mcpp_neighborhood",
            how="left",
        )
    )

    choropleth["offense_count"] = (
        choropleth["offense_count"]
        .fillna(0)
        .astype(int)
    )

    choropleth["crime_rate_per_100k"] = np.where(
        choropleth["population"].notna()
        & (choropleth["population"] > 0),
        (
            choropleth["offense_count"]
            / choropleth["population"]
            * 100_000
        ),
        np.nan,
    )

    choropleth["rate_eligible"] = (
        choropleth["population"]
        >= RATE_MIN_POPULATION
    )

    if "mcpp_neighborhood_display" not in choropleth.columns:
        choropleth["mcpp_neighborhood_display"] = (
            choropleth["mcpp_neighborhood"]
            .astype("string")
            .str.title()
        )

    choropleth["population_display"] = (
        choropleth["population"]
        .map(
            lambda value:
                f"{value:,.0f}"
                if pd.notna(value)
                else "Not available"
        )
    )

    choropleth["rate_display"] = np.where(
        choropleth["rate_eligible"],
        choropleth["crime_rate_per_100k"].map(
            lambda value:
                f"{value:,.1f}"
                if pd.notna(value)
                else "Not available"
        ),
        "Not shown (<5,000 population)",
    )

    total_offenses = int(
        records[EVENT_ID_COLUMN].nunique()
    )

    assigned_offenses = int(
        assignments[EVENT_ID_COLUMN].nunique()
    )

    summary = {
        "total_offenses": total_offenses,
        "assigned_offenses": assigned_offenses,
        "unassigned_offenses": (
            total_offenses - assigned_offenses
        ),
    }

    return choropleth, summary

In [ ]:
def make_v11_crime_map_figure(
    context,
    analysis_state,
    metric_mode="raw",
    layer_mode="choropleth",
):
    analytical_source = context["valid_time"].copy()
    point_source = prepare_v11_point_source(context)

    # --------------------------------------------------
    # CHOROPLETH ANALYTICAL POPULATION
    # --------------------------------------------------
    # Neighborhood selection is deliberately removed here.
    #
    # We need every polygon to remain present so a disabled
    # neighborhood can still be Ctrl+clicked to re-enable it.
    #
    # Date/category/subcategory filters still apply.
    choropleth_state = {
        **analysis_state,
        "neighborhoods": [],
    }

    choropleth_records = filter_crime_records(
        analytical_source,
        choropleth_state,
    )

    # Full analytical selection, including Neighborhood.
    active_records = filter_crime_records(
        analytical_source,
        analysis_state,
    )

    # Point layer obeys all global controls.
    selected_points = filter_crime_records(
        point_source,
        analysis_state,
    )

    choropleth, _ = prepare_v11_choropleth_data(
        context,
        choropleth_records,
    )

    # --------------------------------------------------
    # ACTIVE / DISABLED NEIGHBORHOODS
    # --------------------------------------------------
    all_neighborhoods = sorted(
        set(
            choropleth["mcpp_neighborhood"]
            .dropna()
            .astype("string")
            .tolist()
        )
    )

    selected_neighborhoods = (
        analysis_state.get("neighborhoods")
        or []
    )

    if selected_neighborhoods:
        active_neighborhoods = set(
            normalize_neighborhood_name(
                pd.Series(selected_neighborhoods)
            )
            .dropna()
            .astype(str)
        )
    else:
        # Empty global Neighborhood control means:
        # all neighborhoods enabled.
        active_neighborhoods = set(
            all_neighborhoods
        )

    choropleth["is_active"] = (
        choropleth["mcpp_neighborhood"]
        .isin(active_neighborhoods)
    )

    choropleth["region_status"] = np.where(
        choropleth["is_active"],
        "Enabled",
        "Disabled",
    )

    # --------------------------------------------------
    # SUMMARY
    # --------------------------------------------------
    total_offenses = int(
        active_records[EVENT_ID_COLUMN]
        .nunique()
    )

    assigned_offenses = int(
        active_records.loc[
            active_records[
                "mcpp_neighborhood"
            ].notna(),
            EVENT_ID_COLUMN,
        ]
        .nunique()
    )

    point_offenses = int(
        selected_points[EVENT_ID_COLUMN]
        .nunique()
    )

    summary = {
        "total_offenses": total_offenses,
        "assigned_offenses": assigned_offenses,
        "unassigned_offenses": (
            total_offenses
            - assigned_offenses
        ),
        "point_offenses": point_offenses,
        "not_point_plottable": max(
            total_offenses
            - point_offenses,
            0,
        ),
        "enabled_neighborhoods": len(
            active_neighborhoods
        ),
        "disabled_neighborhoods": (
            len(all_neighborhoods)
            - len(active_neighborhoods)
        ),
    }

    geojson = json.loads(
        choropleth.to_json()
    )

    fig = go.Figure()
    # --------------------------------------------------
    # CHOROPLETH
    # --------------------------------------------------
    if layer_mode in {
        "choropleth",
        "both",
    }:

        if metric_mode == "rate":
            metric_column = "crime_rate_per_100k"
            colorbar_title = "Rate<br>/100k"

            hovertemplate = (
                "<b>%{customdata[0]}</b><br>"
                "Status: %{customdata[4]}<br>"
                "Rate per 100,000: %{customdata[3]}<br>"
                "Offenses: %{customdata[1]:,}<br>"
                "Estimated population: %{customdata[2]}"
                "<br><br>"
                "<i>Ctrl+click to enable/disable</i>"
                "<extra></extra>"
            )

        else:
            metric_column = "offense_count"
            colorbar_title = "Offenses"

            hovertemplate = (
                "<b>%{customdata[0]}</b><br>"
                "Status: %{customdata[4]}<br>"
                "Offenses: %{customdata[1]:,}<br>"
                "Estimated population: %{customdata[2]}<br>"
                "Rate per 100,000: %{customdata[3]}"
                "<br><br>"
                "<i>Ctrl+click to enable/disable</i>"
                "<extra></extra>"
            )

        # --------------------------------------------------
        # PER-REGION VISIBILITY
        # --------------------------------------------------
        # Directly control polygon opacity instead of relying
        # on Plotly's selected/unselected subsystem.
        if metric_mode == "rate":
            region_opacity = np.where(
                ~choropleth["is_active"],
                0.04,
                np.where(
                    choropleth["rate_eligible"],
                    0.70,
                    0.08,
                ),
            )
        else:
            region_opacity = np.where(
                choropleth["is_active"],
                0.70,
                0.04,
            )

        # --------------------------------------------------
        # COLOR DOMAIN
        # --------------------------------------------------
        # Only enabled/eligible regions influence the visible
        # analytical color scale.
        active_metric_mask = choropleth["is_active"].copy()

        if metric_mode == "rate":
            active_metric_mask &= choropleth["rate_eligible"]

        active_metric_values = (
            choropleth.loc[
                active_metric_mask,
                metric_column,
            ]
            .dropna()
        )

        if active_metric_values.empty:
            zmin = None
            zmax = None
        else:
            zmin = float(active_metric_values.min())
            zmax = float(active_metric_values.max())

            # Plotly needs a non-zero color domain.
            if zmin == zmax:
                zmax = zmin + 1

        # Keep every polygon in one trace.
        #
        # For rate-ineligible neighborhoods, a numeric placeholder
        # keeps the polygon present/clickable, while very low opacity
        # prevents it from visually representing a comparative rate.
        if metric_mode == "rate":
            display_z = (
                choropleth[metric_column]
                .fillna(0)
            )
        else:
            display_z = choropleth[metric_column]

        fig.add_trace(
            go.Choroplethmap(
                geojson=geojson,

                locations=(
                    choropleth[
                        "plot_feature_id"
                    ]
                ),

                z=display_z,

                featureidkey=(
                    "properties.plot_feature_id"
                ),

                colorscale="Reds",

                zmin=zmin,
                zmax=zmax,

                marker={
                    # IMPORTANT:
                    # one opacity value per polygon
                    "opacity": region_opacity,

                    "line": {
                        "width": 0.6,
                        "color": (
                            "rgba(255,255,255,0.30)"
                        ),
                    },
                },

                colorbar={
                    "title": colorbar_title,
                    "thickness": 12,
                    "len": 0.55,
                    "x": 0.98,
                },

                customdata=(
                    choropleth[
                        [
                            "mcpp_neighborhood_display",
                            "offense_count",
                            "population_display",
                            "rate_display",
                            "region_status",
                            "mcpp_neighborhood",
                        ]
                    ]
                    .to_numpy()
                ),

                hovertemplate=hovertemplate,

                showlegend=False,
                name="Neighborhoods",
            )
        )

    # --------------------------------------------------
    # POINTS
    # --------------------------------------------------
    if layer_mode in {
        "points",
        "both",
    }:
        point_df = (
            selected_points.copy()
        )

        point_df[TIME_COLUMN] = (
            pd.to_datetime(
                point_df[TIME_COLUMN],
                errors="coerce",
            )
        )

        point_df[
            "offense_time_display"
        ] = (
            point_df[TIME_COLUMN]
            .dt.strftime(
                "%b %d, %Y %H:%M"
            )
            .fillna("Not available")
        )

        point_df[
            "crime_category_display"
        ] = (
            point_df[CATEGORY_COLUMN]
            .astype("string")
            .str.title()
            .fillna("Not available")
        )

        point_df[
            "crime_subcategory_display"
        ] = (
            point_df[SUB_CATEGORY_COLUMN]
            .astype("string")
            .str.title()
            .fillna("Not available")
        )

        point_df[
            "neighborhood_display"
        ] = (
            point_df[
                "mcpp_neighborhood"
            ]
            .astype("string")
            .str.title()
            .fillna("Unassigned")
            .replace(
                "<NA>",
                "Unassigned",
            )
        )

        if (
            "block_address"
            in point_df.columns
        ):
            point_df[
                "block_address_display"
            ] = (
                point_df[
                    "block_address"
                ]
                .astype("string")
                .fillna(
                    "Not available"
                )
                .replace(
                    "<NA>",
                    "Not available",
                )
            )
        else:
            point_df[
                "block_address_display"
            ] = "Not available"

        point_df[
            "report_number_display"
        ] = (
            point_df[ROW_ID_COLUMN]
            .astype("string")
            .fillna("Not available")
            .replace(
                "<NA>",
                "Not available",
            )
        )

        for category in (
            CANONICAL_CRIME_TYPES
        ):
            category_points = (
                point_df[
                    point_df[
                        CATEGORY_COLUMN
                    ]
                    == category
                ]
                .copy()
            )

            if (
                category_points.empty
            ):
                continue

            # MapLibre trace: Scattermap,
            # NOT Scattermapbox.
            fig.add_trace(
                go.Scattermap(
                    lat=(
                        category_points[
                            LAT_COL
                        ]
                    ),
                    lon=(
                        category_points[
                            LON_COL
                        ]
                    ),
                    mode="markers",

                    marker={
                        "size": 7,
                        "opacity": 0.72,
                        "color": (
                            CRIME_CATEGORY_COLOR_MAP
                            .get(
                                category,
                                "#dddddd",
                            )
                        ),
                    },

                    name=category.title(),

                    customdata=(
                        category_points[
                            [
                                "crime_subcategory_display",
                                "crime_category_display",
                                "offense_time_display",
                                "neighborhood_display",
                                "block_address_display",
                                "report_number_display",
                            ]
                        ]
                        .to_numpy()
                    ),

                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        "Category: "
                        "%{customdata[1]}<br>"
                        "Offense time: "
                        "%{customdata[2]}<br>"
                        "Neighborhood: "
                        "%{customdata[3]}<br>"
                        "Block: "
                        "%{customdata[4]}<br>"
                        "Report: "
                        "%{customdata[5]}"
                        "<extra></extra>"
                    ),
                )
            )

    # --------------------------------------------------
    # MAPLIBRE LAYOUT
    # --------------------------------------------------
    fig.update_layout(
        map={
            "style": PLOTLY_MAP_STYLE,
            "center": (
                PLOTLY_SEATTLE_CENTER
            ),
            "zoom": 10,
        },

        paper_bgcolor="#181818",
        plot_bgcolor="#181818",

        font={
            "color": "#dddddd",
        },

        height=690,

        margin={
            "l": 0,
            "r": 0,
            "t": 0,
            "b": 0,
        },

        legend={
            "x": 0.02,
            "y": 0.98,
            "xanchor": "left",
            "yanchor": "top",
            "bgcolor": (
                "rgba(17,17,17,0.80)"
            ),
            "bordercolor": "#333333",
            "borderwidth": 1,
            "font": {
                "size": 10,
                "color": "#ffffff",
            },
        },

        # We want click events but do not want
        # ordinary clicks to invoke Plotly's own
        # selection logic.
        clickmode="event",

        uirevision=(
            "v11-crime-map-camera"
        ),
    )

    return fig, summary

In [52]:
prototype_point_source = (
    prepare_v11_point_source(
        crime_context
    )
)

fallback_points = prototype_point_source[
    prototype_point_source[
        "spatial_mcpp_neighborhood"
    ].isna()
    & prototype_point_source[
        "mcpp_neighborhood"
    ].notna()
]

prototype_records = filter_crime_records(
    crime_context["valid_time"],
    default_state,
)

prototype_choropleth, prototype_summary = (
    prepare_v11_choropleth_data(
        crime_context,
        prototype_records,
    )
)

print(
    "Coordinate-valid offenses rescued by "
    "analytical neighborhood fallback:",
    fallback_points[
        EVENT_ID_COLUMN
    ].nunique(),
)

print(prototype_summary)

assert (
    prototype_choropleth[
        "offense_count"
    ].sum()
    == prototype_summary[
        "assigned_offenses"
    ]
)

assert (
    prototype_summary["total_offenses"]
    ==
    prototype_summary["assigned_offenses"]
    + prototype_summary["unassigned_offenses"]
)

print(
    "v1.1 choropleth reconciliation checks passed."
)

Coordinate-valid offenses rescued by analytical neighborhood fallback: 491
{'total_offenses': 40, 'assigned_offenses': 40, 'unassigned_offenses': 0}


AssertionError: 

In [53]:
map_app = Dash(
    __name__,
    assets_folder=str(
        REPO_ROOT / "assets"
    ),
)


MAP_CARD_STYLE = {
    "background": "#181818",
    "border": "1px solid #333333",
    "borderRadius": "8px",
    "overflow": "hidden",
}


map_app.layout = html.Div(
    children=[
        make_analysis_controls(
            default_state,
            crime_category_options,
            default_categories,
            crime_subcategory_options,
            crime_neighborhood_options,
            analysis_start,
            analysis_end,
        ),

        dcc.Store(
            id="prototype-map-region-toggle",
            data=None,
        ),

        html.Div(
            id="prototype-map-listener-anchor",
            style={"display": "none"},
        ),

        html.Div(
            children=[
                # --------------------------------------
                # MAP HEADER
                # --------------------------------------
                html.Div(
                    children=[
                        html.Div(
                            "Crime Geography",
                            style={
                                "color": "#bbbbbb",
                                "fontSize": "11px",
                                "fontWeight": "600",
                                "letterSpacing": "0.04em",
                                "textTransform": "uppercase",
                            },
                        ),

                        html.Div(
                            children=[
                                dcc.RadioItems(
                                    id="prototype-map-metric",
                                    options=[
                                        {
                                            "label": "Raw",
                                            "value": "raw",
                                        },
                                        {
                                            "label": "Rate /100k",
                                            "value": "rate",
                                        },
                                    ],
                                    value="raw",
                                    inline=True,
                                    labelStyle={
                                        "marginLeft": "12px",
                                        "cursor": "pointer",
                                        "color": "#ffffff",
                                    },
                                    inputStyle={
                                        "marginRight": "4px",
                                    },
                                    style={
                                        "fontSize": "10px",
                                        "color": "#ffffff",
                                    },
                                ),

                                dcc.RadioItems(
                                    id="prototype-map-layer",
                                    options=[
                                        {
                                            "label": "Neighborhoods",
                                            "value": "choropleth",
                                        },
                                        {
                                            "label": "Points",
                                            "value": "points",
                                        },
                                        {
                                            "label": "Both",
                                            "value": "both",
                                        },
                                    ],
                                    value="choropleth",
                                    inline=True,
                                    labelStyle={
                                        "marginLeft": "12px",
                                        "cursor": "pointer",
                                        "color": "#ffffff",
                                    },
                                    inputStyle={
                                        "marginRight": "4px",
                                    },
                                    style={
                                        "fontSize": "10px",
                                        "color": "#ffffff",
                                    },
                                ),
                            ],
                            style={
                                "display": "flex",
                                "alignItems": "center",
                                "gap": "20px",
                            },
                        ),
                    ],
                    style={
                        "display": "flex",
                        "alignItems": "center",
                        "justifyContent": "space-between",
                        "padding": "12px 16px",
                        "borderBottom": "1px solid #333333",
                    },
                ),

                dcc.Graph(
                    id="prototype-crime-map",
                    config={
                        "responsive": True,
                        "displaylogo": False,
                    },
                    style={
                        "height": "690px",
                    },
                ),

                html.Div(
                    id="prototype-map-note",
                    style={
                        "padding": "9px 14px 11px",
                        "borderTop": "1px solid #333333",
                        "color": "#888888",
                        "fontSize": "10px",
                        "lineHeight": "15px",
                    },
                ),
            ],
            style=MAP_CARD_STYLE,
        ),
    ],
    style={
        "background": "#111111",
        "minHeight": "100vh",
        "padding": "20px",
        "fontFamily": "Arial, sans-serif",
    },
)

map_app.clientside_callback(
    """
    function(figure) {
        window.setTimeout(function() {
            const container =
                document.getElementById(
                    "prototype-crime-map"
                );

            if (!container) {
                return;
            }

            const gd =
                container.querySelector(
                    ".js-plotly-plot"
                );

            if (!gd) {
                return;
            }

            // Avoid registering duplicate listeners
            // every time the figure redraws.
            if (gd.__v11RegionToggleBound) {
                return;
            }

            gd.__v11RegionToggleBound = true;

            gd.on(
                "plotly_click",
                function(eventData) {
                    if (
                        !eventData ||
                        !eventData.event
                    ) {
                        return;
                    }

                    // Windows/Linux: Ctrl
                    // macOS equivalent: Command
                    const modifierPressed =
                        eventData.event.ctrlKey ||
                        eventData.event.metaKey;

                    if (!modifierPressed) {
                        return;
                    }

                    if (
                        !eventData.points ||
                        eventData.points.length === 0
                    ) {
                        return;
                    }

                    const point =
                        eventData.points[0];

                    // Only neighborhood polygons should
                    // affect the global neighborhood filter.
                    if (
                        !point.data ||
                        point.data.type !==
                            "choroplethmap"
                    ) {
                        return;
                    }

                    if (
                        !point.customdata ||
                        point.customdata.length < 6
                    ) {
                        return;
                    }

                    const neighborhood =
                        point.customdata[5];

                    if (!neighborhood) {
                        return;
                    }

                    dash_clientside.set_props(
                        "prototype-map-region-toggle",
                        {
                            data: {
                                neighborhood:
                                    neighborhood,
                                timestamp:
                                    Date.now(),
                            },
                        }
                    );
                }
            );
        }, 0);

        return dash_clientside.no_update;
    }
    """,

    Output(
        "prototype-map-listener-anchor",
        "children",
    ),

    Input(
        "prototype-crime-map",
        "figure",
    ),

    prevent_initial_call=False,
)

In [ ]:
@map_app.callback(
    Output(
        "prototype-crime-map",
        "figure",
    ),
    Output(
        "prototype-map-note",
        "children",
    ),
    Input(
        "crime-analysis-start-date-input",
        "value",
    ),
    Input(
        "crime-analysis-end-date-input",
        "value",
    ),
    Input(
        "crime-category-filter",
        "value",
    ),
    Input(
        "crime-subcategory-filter",
        "value",
    ),
    Input(
        "crime-neighborhood-filter",
        "value",
    ),
    Input(
        "prototype-map-metric",
        "value",
    ),
    Input(
        "prototype-map-layer",
        "value",
    ),
)
def update_v11_crime_map(
    start_date,
    end_date,
    categories,
    subcategories,
    neighborhoods,
    metric_mode,
    layer_mode,
):
    bounded_dates = validate_analysis_dates(
        start_date,
        end_date,
        analysis_start,
        analysis_end,
        clamp=True,
    )

    if bounded_dates is None:
        return (
            go.Figure(),
            "Invalid analysis period.",
        )

    current_start, current_end = bounded_dates

    state = {
        "start_date": current_start,
        "end_date": current_end,
        "crime_categories": categories or [],
        "crime_subcategories": subcategories or [],
        "neighborhoods": neighborhoods or [],
    }

    fig, summary = make_v11_crime_map_figure(
        context=crime_context,
        analysis_state=state,
        metric_mode=metric_mode,
        layer_mode=layer_mode,
    )

    note = (
        f"{summary['total_offenses']:,} selected offenses"
        f"  •  "
        f"{summary['assigned_offenses']:,} assigned to an MCPP"
        f"  •  "
        f"{summary['unassigned_offenses']:,} unassigned"
        f"  •  "
        f"{summary['point_offenses']:,} point-plottable"
    )

    if metric_mode == "rate":
        note += (
            "  •  Estimated rate per 100,000 residents; "
            "MCPPs below 5,000 estimated residents are "
            "not shaded in rate mode. Rates reflect the "
            "selected period and are not annualized."
        )

    return fig, note
@map_app.callback(
    Output(
        "crime-neighborhood-filter",
        "value",
    ),
    Input(
        "prototype-map-region-toggle",
        "data",
    ),
    State(
        "crime-neighborhood-filter",
        "value",
    ),
    State(
        "crime-neighborhood-filter",
        "options",
    ),
    prevent_initial_call=True,
)
def toggle_neighborhood_from_map(
    toggle_data,
    current_value,
    neighborhood_options,
):
    if not toggle_data:
        raise PreventUpdate

    neighborhood = (
        toggle_data.get(
            "neighborhood"
        )
    )

    if not neighborhood:
        raise PreventUpdate

    all_neighborhoods = [
        option["value"]
        for option
        in neighborhood_options
    ]

    # mark 

    all_set = set(
        all_neighborhoods
    )

    if neighborhood not in all_set:
        raise PreventUpdate

    # Existing global-control convention:
    #
    # [] = all neighborhoods.
    #
    # Materialize that implicit "all"
    # selection the first time a region
    # is disabled.
    if not current_value:
        enabled = set(
            all_neighborhoods
        )
    else:
        enabled = set(
            current_value
        )

    if neighborhood in enabled:
        enabled.remove(
            neighborhood
        )
    else:
        enabled.add(
            neighborhood
        )

    # Do not allow a map interaction to
    # disable every neighborhood.
    if not enabled:
        raise PreventUpdate

    # If everything has been re-enabled,
    # collapse back to [] so the production
    # control retains its existing
    # "All neighborhoods" semantics.
    if enabled == all_set:
        return []

    # Preserve the production dropdown's
    # option order rather than returning
    # arbitrary set order.
    return [
        neighborhood_name
        for neighborhood_name
        in all_neighborhoods
        if neighborhood_name
        in enabled
    ]

In [55]:
map_app.run(
    jupyter_mode="inline",
    debug=False,
    port=8054,
)

In [56]:
diagnostic_records = prototype_records.copy()

diagnostic_records["mcpp_neighborhood"] = (
    normalize_neighborhood_name(
        diagnostic_records["mcpp_neighborhood"]
    )
)

diagnostic_boundaries = (
    crime_context["mcpp_boundaries"]
    .copy()
)

diagnostic_boundaries["mcpp_neighborhood"] = (
    normalize_neighborhood_name(
        diagnostic_boundaries["mcpp_neighborhood"]
    )
)


# ---------------------------------------------
# 1. Boundary uniqueness
# ---------------------------------------------
boundary_counts = (
    diagnostic_boundaries[
        "mcpp_neighborhood"
    ]
    .value_counts()
)

duplicate_boundary_names = (
    boundary_counts[
        boundary_counts > 1
    ]
)

print("Boundary rows:", len(diagnostic_boundaries))

print(
    "Unique boundary neighborhoods:",
    diagnostic_boundaries[
        "mcpp_neighborhood"
    ].nunique(),
)

print("\nDuplicate boundary neighborhood names:")
display(duplicate_boundary_names)


# ---------------------------------------------
# 2. Analytical vs boundary vocabulary
# ---------------------------------------------
analytical_names = set(
    diagnostic_records[
        "mcpp_neighborhood"
    ]
    .dropna()
    .astype(str)
)

boundary_names = set(
    diagnostic_boundaries[
        "mcpp_neighborhood"
    ]
    .dropna()
    .astype(str)
)

analytical_only = sorted(
    analytical_names
    - boundary_names
)

boundary_only = sorted(
    boundary_names
    - analytical_names
)

print(
    "\nAnalytical neighborhood names not represented "
    "by an MCPP polygon:"
)

print(analytical_only)

print(
    "\nBoundary neighborhoods with no selected offenses:"
)

print(boundary_only)


# ---------------------------------------------
# 3. Exact count reconciliation
# ---------------------------------------------
assigned_ids = (
    diagnostic_records.loc[
        diagnostic_records[
            "mcpp_neighborhood"
        ].notna(),
        EVENT_ID_COLUMN,
    ]
    .nunique()
)

mapped_to_known_mcpp_ids = (
    diagnostic_records.loc[
        diagnostic_records[
            "mcpp_neighborhood"
        ].isin(boundary_names),
        EVENT_ID_COLUMN,
    ]
    .nunique()
)

choropleth_sum = int(
    prototype_choropleth[
        "offense_count"
    ].sum()
)

print("\nAssigned unique offenses:", assigned_ids)

print(
    "Assigned to a recognized MCPP:",
    mapped_to_known_mcpp_ids,
)

print(
    "Sum across choropleth polygon rows:",
    choropleth_sum,
)

Boundary rows: 58
Unique boundary neighborhoods: 58

Duplicate boundary neighborhood names:


Series([], Name: count, dtype: int64[pyarrow])


Analytical neighborhood names not represented by an MCPP polygon:
['-']

Boundary neighborhoods with no selected offenses:
['alaska junction', 'alki', 'ballard south', 'brighton/dunlap', 'claremont/rainier vista', 'commercial duwamish', 'commercial harbor island', 'eastlake - east', 'eastlake - west', 'first hill', 'genesee', 'georgetown', 'greenwood', 'high point', 'highland park', 'hillman city', 'judkins park/north beacon hill', 'lakewood/seward park', 'madison park', 'madrona/leschi', 'mid beacon hill', 'miller park', 'montlake/portage bay', 'morgan', 'new holly', 'north admiral', 'north beacon hill', 'north delridge', 'northgate', 'phinney ridge', 'pigeon point', 'pioneer square', 'rainier beach', 'roxhill/westwood/arbor heights', 'sandpoint', 'sodo', 'south beacon hill', 'south park', 'wallingford']

Assigned unique offenses: 40
Assigned to a recognized MCPP: 37
Sum across choropleth polygon rows: 37


## 13. Crime daily time series (already exists/doesn't need significant revision)

**Authoritative source:** `crime_context["valid_time"]`

For dashboard-identical preparation, use `prepare_daily_event_data()` from `dashboard.crime_dashboard_figures` rather than manually resampling.

That helper handles the retained analysis domain and zero-filled daily series used by the figure.

In [ ]:
from dashboard.crime_dashboard_figures import prepare_daily_event_data

selected_categories = sorted(
    crime_context["valid_time"]["offense_category"].dropna().unique()
)

daily_crime, crime_window = prepare_daily_event_data(
    crime_context,
    selected_categories,
    state,
)

daily_crime.head(), crime_window

(        date  reported_offenses  unique_reports  rolling_7_day_avg
 0 2025-09-13                188             167                NaN
 1 2025-09-14                180             157                NaN
 2 2025-09-15                199             179                NaN
 3 2025-09-16                191             173                NaN
 4 2025-09-17                181             163                NaN,
 {'earliest_available_day': Timestamp('2024-09-09 00:00:00'),
  'latest_available_day': Timestamp('2026-09-13 00:00:00'),
  'earliest_analysis_day': Timestamp('2025-09-13 00:00:00'),
  'plot_start_day': Timestamp('2025-09-13 00:00:00'),
  'plot_end_day': Timestamp('2026-09-13 00:00:00'),
  'initial_view_start': Timestamp('2026-09-12 00:00:00')})

## 14. Population access and provenance

Both the crime and calls contexts expose:

- `neighborhood_population`
- `city_population`
- `population_metadata`

The same values can be loaded directly with `load_dashboard_population()`.

Use:

- **direct city population** for Seattle-wide rates
- calibrated neighborhood `population` for MCPP rates
- `population_metadata` to display/record ACS vintage and methodology

Do not use `population_raw` as the runtime denominator.

In [ ]:
display_columns = [
    "mcpp_neighborhood",
    "population",
    "population_raw",
    "population_year",
    "source_vintage",
    "estimation_method",
]

neighborhood_population[
    [c for c in display_columns if c in neighborhood_population.columns]
].head()

,mcpp_neighborhood,population,population_raw,population_year,source_vintage,estimation_method
0,alaska junction,16244.0,16243.0,2024,2024,ACS block-group population distributed using 2...
1,alki,7729.0,7728.0,2024,2024,ACS block-group population distributed using 2...
2,ballard north,30350.0,30348.0,2024,2024,ACS block-group population distributed using 2...
3,ballard south,26048.0,26046.0,2024,2024,ACS block-group population distributed using 2...
4,belltown,11501.0,11500.0,2024,2024,ACS block-group population distributed using 2...


## 15. Quick collaborator reference

| Component | Authoritative data access | Assignee |
| --- | --- | --- |
| Overall crime count | `crime_context["valid_time"]` | Ben Carr |
| Overall crime rate | `crime_context["valid_time"]` + `crime_context["city_population"]` | Parfait Ngandu |
| Crime category counts | `crime_context["valid_time"]` | Ben Carr |
| Median qualified response | `calls_context["response_analysis"]` | Ben Carr |
| UOF incidents | `uof_context["df"]` + `count_uof_incidents()` | Parfait Ngandu |
| OIS events | `uof_context["ois_events"]` + `count_ois_events()` | Parfait Ngandu |
| Neighborhood crime ranking | `crime_context["valid_time"]` + neighborhood population | Parfait Ngandu |
| Neighborhood response ranking | `calls_context["response_analysis"]` | Ben Carr |
| Crime choropleth | `crime_context["valid_time"]` + boundaries + population | Ben Carr |
| Crime point map | `crime_context["event_mcpp"]` | Ben Carr |
| Crime daily series | `crime_context["valid_time"]` via `prepare_daily_event_data()` | Ben Carr |




### Rule of thumb

- **Analysis/KPIs:** prefer `valid_time` or `response_analysis`.
- **Map points:** use `event_mcpp`.
- **QA/classification inspection:** use `df`.
- **Rates:** pair the analytical numerator with the appropriate production population denominator.
- **Do not substitute coordinate-valid records for the full analytical population.**

## 16. Optional sanity checks after loading

These checks are useful for collaborators to confirm that they are working with the intended production contexts before beginning analysis.

In [ ]:
assert "valid_time" in crime_context
assert "response_analysis" in calls_context
assert "ois_events" in uof_context

assert crime_context["city_population"] > 0
assert calls_context["city_population"] > 0

assert crime_context["valid_time"]["offense_id"].notna().all()
assert calls_context["response_analysis"]["cad_event_number"].notna().all()

print("Core collaborator data-access checks passed.")

Core collaborator data-access checks passed.
